### Project: Cafe Sales Data ETL Pipeline

### Goal: Prepare data for DBeaver import and subsequent SQL analysis.

### Main Steps:
* **Cleaning:** Standardizing formats, handling missing values, and ensuring numeric consistency.
* **Preparation:** Casting data types to the format required for MySQL (DBeaver) loading.

In [13]:
import pandas as pd

df_new = df.copy()
df_new['Дата'] = pd.to_datetime(df_new['Дата'], format='%d.%m.%Y %H:%M')
rename_dict = {
    'Дата': 'created_at',
    'Номер чека': 'receipt_id',
    'Тип чека': 'receipt_type',
    'Продажи': 'sales',
    'Скидки': 'discount',
    'Выручка': 'revenue',
    'Налоги': 'tax',
    'Итого': 'total',
    'Себестоимость товаров': 'cost',
    'Валовая прибыль': 'profit',
    'Тип оплаты': 'payment_type',
    'Описание': 'items',
    'Тип заказа': 'order_type',
    'Касса': 'cash_desk',
    'Заведение': 'location',
    'Кассир': 'cashier',
    'Имя клиента': 'client_name',
    'Контакты клиента': 'phone',
    'Статус': 'status'
}

#  Переименовать столбцы
df_new = df_new.rename(columns=rename_dict)

# смотрю правильно ли переименованы столбцы
print(df_new.columns.tolist())
df_new.info()
df_new.to_csv('cafe.csv', index=False, encoding='utf-8')

['created_at', 'receipt_id', 'receipt_type', 'sales', 'discount', 'revenue', 'tax', 'total', 'cost', 'profit', 'payment_type', 'items', 'order_type', 'cash_desk', 'location', 'cashier', 'client_name', 'phone', 'status']
<class 'pandas.DataFrame'>
RangeIndex: 7490 entries, 0 to 7489
Data columns (total 19 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   created_at    7490 non-null   datetime64[us]
 1   receipt_id    7490 non-null   str           
 2   receipt_type  7490 non-null   str           
 3   sales         7490 non-null   float64       
 4   discount      7490 non-null   float64       
 5   revenue       7490 non-null   float64       
 6   tax           7490 non-null   float64       
 7   total         7490 non-null   float64       
 8   cost          7490 non-null   float64       
 9   profit        7490 non-null   float64       
 10  payment_type  7490 non-null   str           
 11  items         7490 non-null  

In [25]:
import pandas as pd
df = pd.read_csv('summary.csv')
df_sum = df.copy() 
df_sum.head()
df_sum.rename(columns={
    'Товар': 'product_name',
    'Артикул': 'sku',
    'Категория': 'category',
    'Товаров продано': 'units_sold',
    'Сумма продаж': 'sales_amount',
    'Товаров возвращено': 'units_returned',
    'Сумма возвратов': 'refund_amount',
    'Сумма скидок': 'discount_amount',
    'Выручка': 'revenue',
    'Себестоимость товаров': 'cost',
    'Валовая прибыль': 'gross_profit',
    'Маржа': 'margin_pct',
    'Сумма налогов': 'tax_amount'
}, inplace=True)
df_sum.info()
df_sum.to_csv('abc.csv', index=False, encoding='utf-8')



<class 'pandas.DataFrame'>
RangeIndex: 178 entries, 0 to 177
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   product_name     178 non-null    str    
 1   sku              178 non-null    int64  
 2   category         178 non-null    str    
 3   units_sold       178 non-null    float64
 4   sales_amount     178 non-null    float64
 5   units_returned   178 non-null    float64
 6   refund_amount    178 non-null    float64
 7   discount_amount  178 non-null    float64
 8   revenue          178 non-null    float64
 9   cost             178 non-null    float64
 10  gross_profit     178 non-null    float64
 11  margin_pct       178 non-null    str    
 12  tax_amount       178 non-null    float64
dtypes: float64(9), int64(1), str(3)
memory usage: 18.2 KB


# Формируем базу стат данных среднего чека по кажому часу каждого дня
WITH stat_base AS(
SELECT
DATE(created_at) AS day,
WEEKDAY(created_at) AS weekday, HOUR(created_at) AS hour,
SUM(sales) / COUNT(DISTINCT receipt_id) AS avg_receipt
FROM cafe
WHERE status = 'Закрыт'
GROUP BY DATE(created_at),
WEEKDAY(created_at), HOUR(created_at)
)
SELECT weekday, hour,
ROUND(AVG(avg_receipt), 2) AS mean_avg_receipt, -- Среднее значение среднего чека 
ROUND(STDDEV(avg_receipt), 2) AS volatility,    -- Насколько сильно "скачет" средний чек по разным дням в один час
round((STDDEV(avg_receipt) / AVG(avg_receipt)), 2) AS Var -- Коэффициент вариации
FROM stat_base
GROUP BY weekday, hour
ORDER BY weekday, hour;

SELECT WEEKDAY(created_at),
SUM(sales), SUM(ABS(profit)), -- посколку есть строки где прибыль отрицательна так как оплата 100% прошла за счет бонусной системы
COUNT(DISTINCT receipt_id),
-- Средний чек: сколько денег приносит один визит
ROUND(SUM(sales) / COUNT(DISTINCT receipt_id), 2) AS avg_revenue_per_receipt,
    -- Эффективность: сколько прибыли приносит один чек ("золотая" метрика)
ROUND(SUM(ABS(profit)) / COUNT(DISTINCT receipt_id), 2) AS profit_per_receipt
FROM cafe
WHERE status = 'Закрыт'
GROUP BY 1
ORDER BY 6 DESC;

#поиск отмен заказов по дням недели
SELECT WEEKDAY(created_at),
SUM(CASE WHEN status = 'Отменен' THEN 1 ELSE 0 END) AS cancel,
ROUND(SUM(CASE WHEN status = 'Отменен' THEN 1 ELSE 0 END) * 100 / COUNT(DISTINCT receipt_id), 2) AS part
FROM cafe
GROUP BY 1
ORDER BY 1;

SELECT 
    HOUR(created_at) AS hour_of_day,
    COUNT(DISTINCT receipt_id) AS total_receipts,
    -- Средняя прибыль с одного чека в этот час
    ROUND(SUM(ABS(profit)) / COUNT(DISTINCT receipt_id), 2) AS profit_per_receipt
FROM cafe
WHERE status = 'Закрыт'
GROUP BY 1
ORDER BY 1;

-- создаем временную таблицу для визуализации удержания
CREATE TABLE loyal_clients AS
#формируем базу постоянных клиентов зарегистрированных в бонусной системе и ранжируем покупки
WITH clients AS(
SELECT WEEKDAY(created_at) AS weekday, DATE(created_at) AS date_c,
phone, sales, discount, profit,
ROW_NUMBER () OVER(PARTITION BY phone ORDER BY created_at) AS rang
FROM cafe
WHERE phone IS NOT NULL 
AND phone != ''        -- отсекает пустые строки
AND phone != ' '    -- отсекает строки с пробелом
AND status = 'Закрыт')
SELECT weekday, date_c, # считаем разницу в днях между покупками -- считаем накопительный средний чек -- сравниваем текущий заказ с предыдущим
phone, sales, discount, profit, rang,
DATEDIFF(date_c, LAG(date_c) OVER(PARTITION BY phone ORDER BY date_c)) AS diff_days,
AVG(sales) OVER(PARTITION BY phone ORDER BY date_c ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_avg_receipt,
CASE
	WHEN sales > LAG(sales) OVER (PARTITION BY phone ORDER BY date_c) THEN 1
	ELSE 0
END AS progit_growing
FROM clients;

# анализируем динамику поведения постоянных клиентов по порядковому номеру визита
-- rang показывает номер визита клиента. 
-- pct_growing_profit показывает % чеков, которые оказались больше предыдущего.
-- cumulative_avg_check показывает среднюю сумму всех чеков клиента с 1-го по текущий визит.
SELECT
rang, COUNT(*) AS total_customers,
ROUND(AVG(progit_growing) * 100, 2) AS pct_growing_profit,
ROUND(AVG(cum_avg_receipt), 2) AS cumulative_avg_check
FROM loyal_clients
GROUP BY rang 
HAVING total_customers > 10
ORDER BY rang;


SELECT COUNT(DISTINCT phone)
FROM cafe;

# islands -- retention
SELECT phone, COUNT(*) as visit_count, 
ROUND(AVG(sales), 2) as avg_check, 
SUM(profit),
SUM(sales) as total_revenue,
SUM(discount) AS discount,
ROUND(SUM(discount) / SUM(sales) * 100, 2) AS loyalty_cost_percentage
FROM loyal_clients
GROUP BY phone
ORDER BY visit_count DESC;

# islands >= 3 buying days
WITH daily_visits AS (
    -- Сначала убираем дубли: если клиент был 2 раза в день, 
    -- считаем это как один "день активности"
    SELECT DISTINCT phone, date_c
    FROM loyal_clients
),
date_groups AS (
    SELECT 
        phone, 
        date_c,
        -- Теперь используем ROW_NUMBER по уникальным дням
        DATE_SUB(date_c, INTERVAL ROW_NUMBER() OVER(PARTITION BY phone ORDER BY date_c) DAY) as grp
    FROM daily_visits
)
SELECT 
    phone, 
    MIN(date_c) as streak_start, 
    MAX(date_c) as streak_end, 
    COUNT(*) as streak_days
FROM date_groups
GROUP BY phone, grp
HAVING streak_days >= 3
ORDER BY streak_days DESC;

# когда приходят лояльные клиенты
SELECT DAYNAME(date_c), COUNT(*) as visits, SUM(sales) as revenue
FROM loyal_clients lc 
GROUP BY DAYNAME(date_c)
ORDER BY visits DESC;

# когда приходят все клиенты
SELECT DAYNAME(created_at), COUNT(*) as visits, SUM(sales) as revenue
FROM cafe 
GROUP BY DAYNAME(created_at)
ORDER BY visits DESC;

#что постоянные клиенты покупали во второй раз
WITH base AS(
SELECT DATE(created_at), phone, items,
ROW_NUMBER() OVER(PARTITION BY phone ORDER BY created_at) AS rang
FROM cafe)
SELECT *
FROM base
WHERE rang = 2;



#abc analysis

# abc sales_anlysis 
CREATE TABLE abc_sales AS
WITH all_sku AS(
SELECT product_name, sku, category, sales_amount,
sales_amount * 100 / SUM(sales_amount) OVER () AS part_sales
FROM abc
),
part_sales AS(
SELECT product_name, sku, category, sales_amount,
SUM(part_sales) OVER(ORDER BY sales_amount DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_sales_part
FROM all_sku)
SELECT product_name, sku, category, sales_amount, ROUND(cum_sales_part, 2),
CASE 
WHEN cum_sales_part < 80 THEN 'A'
WHEN cum_sales_part < 95 THEN 'B'
ELSE 'C'
END AS abc_sales_category
FROM part_sales;


CREATE TABLE abc_profit AS
# abc profit_anlysis 
WITH all_sku_profit AS(
SELECT product_name, gross_profit,
gross_profit * 100 / SUM(gross_profit) OVER () AS part_profit
FROM abc
),
part_profit AS(
SELECT product_name, gross_profit,
SUM(part_profit) OVER(ORDER BY gross_profit DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_profit_part
FROM all_sku_profit)
SELECT product_name, gross_profit, ROUND(cum_profit_part, 2) AS cum_profit,
CASE 
WHEN cum_profit_part < 80 THEN 'A'
WHEN cum_profit_part < 95 THEN 'B'
ELSE 'C'
END AS abc_profit_category
FROM part_profit;

SELECT product_name, sku, category, sales_amount, abc_sales_category,
abc_profit_category
FROM abc_sales INNER JOIN abc_profit USING(product_name)
ORDER BY 4 DESC;

SELECT MONTH(created_at), SUM(sales)
FROM cafe c 
WHERE location = 'fox Coffee'
GROUP BY MONTH(created_at)
ORDER BY 1;

